# 02. Frequency analysis

Counts the most common keywords and bigrams in the cleaned posts, and tests which words appear more often here than in general English. This is a light first pass at what r/WomensHealth is about, and it fed the keyword list alongside BERTopic.

**Run from `notebooks/Reddit_Data/`.**

**Input:** `data/interim/cleaned_submissions.parquet` (from notebook 01)
**Output:** `data/processed/keyword_frequencies.csv`, `data/processed/keyword_keyness.csv`, `data/processed/bigram_frequencies.csv`

In [ ]:
import pandas as pd
from collections import Counter

df = pd.read_parquet("../../data/interim/cleaned_submissions.parquet")
df.info()

## Single keyword counts

In [ ]:
title_counts = Counter(kw for kws in df["title_keywords"] for kw in kws)
selftext_counts = Counter(kw for kws in df["selftext_keywords"] for kw in kws)
combined_counts = title_counts + selftext_counts

print("Top 50 keywords overall:")
for word, count in combined_counts.most_common(50):
    print(f"{word}: {count}")

In [ ]:
keyword_freq = pd.DataFrame(combined_counts.most_common(), columns=["keyword", "count"])
keyword_freq.to_csv("../../data/processed/keyword_frequencies.csv", index=False)
keyword_freq.head(20)

## Keyness against general English

A raw count only tells us a word is common, not that it is distinctive to this community. "Period" being frequent means nothing on its own until we know how often it turns up in ordinary English. To check that, we compare our counts against expected frequencies from the wordfreq package.

We use a log-likelihood ratio (G-test) rather than a chi-squared test. Chi-squared assumes a normal distribution, and word counts are not normally distributed. Words with a frequency below 1 in 50,000 make up roughly 20 to 30 percent of typical English text, so the distribution has much longer tails and chi-squared overstates significance (Dunning, 1993).

Because we test 50 words at once, we apply a Bonferroni correction to the significance threshold.

In [ ]:
import math
from wordfreq import word_frequency
from scipy.stats import chi2

# wordfreq returns a rate, so we scale it to a reference corpus of one billion tokens
REF_SIZE = 1_000_000_000
n1 = sum(combined_counts.values())  # total tokens in our corpus

TOP_N = 50
alpha = 0.05 / TOP_N  # Bonferroni: split alpha across every word tested
sig_threshold = -math.log10(alpha)

rows = []
for term, o1 in combined_counts.most_common(TOP_N):
    # Floor at 0.01 so words wordfreq has never seen do not divide by zero
    o2 = max(word_frequency(term, "en") * REF_SIZE, 0.01)

    # Expected counts if the word were equally common in both corpora
    e1 = n1 * (o1 + o2) / (n1 + REF_SIZE)
    e2 = REF_SIZE * (o1 + o2) / (n1 + REF_SIZE)

    g2 = 2 * (o1 * math.log(o1 / e1) + o2 * math.log(o2 / e2))

    # logsf keeps precision where p is far too small to represent directly
    neg_log10_p = -chi2.logsf(g2, 1) / math.log(10)

    # G2 measures how far apart the two rates are, not which way. A word can be
    # highly significant because it is rare here, so we record direction separately.
    overused = (o1 / n1) > (o2 / REF_SIZE)

    rows.append({
        "keyword": term,
        "observed": o1,
        "expected_english": round(o2, 2),
        "G2": round(g2, 2),
        "neg_log10_p": round(neg_log10_p, 2),
        "overused": overused,
        "significant": neg_log10_p > sig_threshold,
    })

keyness = pd.DataFrame(rows)

In [ ]:
print(f"Bonferroni threshold: -log10(p) > {sig_threshold:.2f}")
print(f"{keyness['significant'].sum()} of {len(keyness)} keywords significant\n")

for r in rows:
    direction = "over" if r["overused"] else "under"
    flag = "***" if r["significant"] else ""
    print(f"{r['keyword']:15s} obs={r['observed']:<6} G2={r['G2']:<12.1f} "
          f"-log10(p)={r['neg_log10_p']:<8.1f} {direction:5s} {flag}")

In [ ]:
keyness.to_csv("../../data/processed/keyword_keyness.csv", index=False)
keyness.head(20)

## Bigram counts

In [ ]:
def get_bigrams(tokens):
    return list(zip(tokens, tokens[1:]))

# Built from the stopword-filtered keywords so pairs like "yeast infection"
# survive but "of the" does not
df["title_bigrams"] = df["title_keywords"].apply(get_bigrams)
df["selftext_bigrams"] = df["selftext_keywords"].apply(get_bigrams)

title_bigram_counts = Counter(bg for bgs in df["title_bigrams"] for bg in bgs)
selftext_bigram_counts = Counter(bg for bgs in df["selftext_bigrams"] for bg in bgs)
combined_bigram_counts = title_bigram_counts + selftext_bigram_counts

print("Top 50 bigrams:")
for (w1, w2), count in combined_bigram_counts.most_common(50):
    print(f"{w1} {w2}: {count}")

In [ ]:
bigram_freq = pd.DataFrame(
    [(f"{w1} {w2}", count) for (w1, w2), count in combined_bigram_counts.most_common()],
    columns=["bigram", "count"],
)
bigram_freq.to_csv("../../data/processed/bigram_frequencies.csv", index=False)
bigram_freq.head(20)